# 04 · The same questions in SQL, at warehouse scale

**Produces:** the clean tables loaded into a database, and the outputs of `sql/03_profile.sql`, `sql/04_transformations.sql`, `sql/05_performance.sql`.

**Needs:** notebook 03 run; either a PostgreSQL connection or nothing (the notebook falls back to SQLite from the standard library).

Why SQL after pandas: the warehouse is where finance will re-run your numbers, the logical order of a query (FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY) is the interview question that never goes away, and window functions answer "compared with the previous month, per city" in one pass where pandas needs three.

## CONFIG

In [5]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

import sqlite3, os
PG_DSN = os.environ.get("KALPA_PG")          # e.g. "host=localhost dbname=kalpa user=postgres"; leave unset for SQLite
DB_PATH = CONFIG["clean"] / "kalpa.sqlite"
SQL_DIR = ROOT / "sql"

3.0.6 2.5.4


## 1 · DO: load the clean tables

In [6]:
tables = {
    "bookings": pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["created_at", "slot_at", "collected_at"], dtype={"rescheduled_from": "string", "status_at_export": "string"}),
    "patients": pd.read_csv(CONFIG["clean"] / "patients_clean.csv.gz", dtype="string"),
    "invoices": pd.read_csv(CONFIG["clean"] / "invoices_clean.csv.gz", parse_dates=["invoice_date"]),
    "payments": pd.read_csv(CONFIG["clean"] / "payments_clean.csv.gz", parse_dates=["created_at"]),
    "plan": pd.read_csv(CONFIG["raw"] / "plan" / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="target_tests"),
}
tables["plan"]["city"] = tables["plan"]["city"].replace({"Bangalore": "Bengaluru"})
if PG_DSN:
    import psycopg2
    con = psycopg2.connect(PG_DSN)
    print("PostgreSQL:", con.server_version)
else:
    if DB_PATH.exists(): DB_PATH.unlink()
    con = sqlite3.connect(DB_PATH)
    print("SQLite at", DB_PATH)
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)
    print(f"{name}: {len(df):,} rows")

SQLite at /home/chaitu/Kalpa_Health_Project_Group_No_4/data/clean/kalpa.sqlite
bookings: 563,380 rows
patients: 303,153 rows
invoices: 470,776 rows
payments: 473,507 rows
plan: 36 rows


## 2 · DO: run the SQL files

The files in `sql/` are written for PostgreSQL. `03_profile.sql` and `04_transformations.sql` also run on SQLite with the small substitutions `run_sql` makes; `05_performance.sql` (indexes and EXPLAIN) is PostgreSQL only and is skipped on SQLite.

In [7]:
def run_sql(path, con):
    text = pathlib.Path(path).read_text()
    is_sqlite = isinstance(con, sqlite3.Connection)
    outputs = []
    for raw in text.split(";\n"):
        lines = [ln for ln in raw.strip().splitlines() if ln.strip()]
        title = next((ln.lstrip("- ").strip() for ln in lines if ln.strip().startswith("--")), "")[:90]
        stmt = "\n".join(ln for ln in lines if not ln.strip().startswith("--") and not ln.strip().startswith("\\"))
        if not stmt.strip():
            continue
        if is_sqlite:
            stmt = (stmt.replace("date_trunc('month', ", "strftime('%Y-%m', ").replace("::date", "").replace("::numeric", "")
                        .replace("EXTRACT(YEAR FROM ", "strftime('%Y', "))
            if "EXPLAIN" in stmt.upper() or "CREATE INDEX" in stmt.upper() or stmt.upper().startswith("ANALYZE"):
                continue
        try:
            outputs.append((title or stmt.splitlines()[0][:70], pd.read_sql_query(stmt, con)))
        except Exception as err:
            outputs.append((title or stmt.splitlines()[0][:70], pd.DataFrame({"error": [str(err)[:200]]})))
    return outputs

for title, df in run_sql(SQL_DIR / "03_profile.sql", con):
    print("\n==", title); print(df.head(12).to_string(index=False))


== Profile in SQL: the same questions as notebook 01, asked of the loaded tables.
     tbl  rows_  distinct_keys          first_slot           last_slot
bookings 563380         563380 2025-01-01 06:00:00 2026-07-04 10:50:00

== Null rates per column, the long way: one line per column so the reader sees the shape of t
         col  null_rate
collected_at     0.1735
   person_id     0.0000
      status     0.0000

== Grain check: rows per booking must be 1 here; if it is not, the load doubled something
 max_rows_per_booking
                    1

== Bookings per system per month: the migration is visible as a change of system, not of volu
system   month  bookings  tests
legacy 2025-01     31902  73094
legacy 2025-02     31528  72551
legacy 2025-03     32142  73958
legacy 2025-04     31296  71974
legacy 2025-05     31166  71153
legacy 2025-06     30769  70881
legacy 2025-07     30687  70659
legacy 2025-08     29501  67622
legacy 2025-09     31129  71346
legacy 2025-10     30098  69348
le

In [8]:
for title, df in run_sql(SQL_DIR / "04_transformations.sql", con):
    print("\n==", title); print(df.head(12).to_string(index=False))


== The methods, as CTE pipelines a finance analyst can read top to bottom.
   half  tests  visits  people  visits_per_person  tests_per_visit
H1_2025 433611  188803   94275             2.0027           2.2966
H1_2026 455383  194020   97827             1.9833           2.3471

== 2. Monthly rollup with window functions. Grain: one row per city per month.
     city   month  tests  prev_month  mom_pct  running_total  rank_in_month
Ahmedabad 2025-01  12728         NaN      NaN          12728              3
Ahmedabad 2025-02  12476     12728.0     -2.0          25204              3
Ahmedabad 2025-03  12462     12476.0     -0.1          37666              3
Ahmedabad 2025-04  13160     12462.0      5.6          50826              3
Ahmedabad 2025-05  12659     13160.0     -3.8          63485              3
Ahmedabad 2025-06  12995     12659.0      2.7          76480              3
Ahmedabad 2025-07  12368     12995.0     -4.8          88848              3
Ahmedabad 2025-08  11724     12368.

> **Interview angle.** Write the logical order in which a SELECT with a GROUP BY and a HAVING is evaluated, and use it to explain why a column alias from SELECT cannot be used in WHERE.
>
> *What a good answer covers:* FROM and JOIN, then WHERE, then GROUP BY, then HAVING, then SELECT, then ORDER BY and LIMIT. WHERE runs before SELECT has produced the alias, so the alias does not exist yet; HAVING runs after grouping so it can see aggregates; ORDER BY runs after SELECT so it can see aliases.

> **Interview angle.** When does a window function beat a GROUP BY, and what defines its result?
>
> *What a good answer covers:* When you need every row kept and a value computed over its neighbours: rank within a city, the previous month's value, a running total. The partition and the order inside the window define everything; a running total is only deterministic when the order has a tiebreaker.

## 3 · DO: performance, measured not guessed (PostgreSQL only)

In [9]:
if PG_DSN:
    for title, df in run_sql(SQL_DIR / "05_performance.sql", con):
        print("\n==", title); print(df.head(20).to_string(index=False))
else:
    print("Skipped on SQLite. On PostgreSQL this file times a monthly rollup before and after an index on (city, slot_at) and prints EXPLAIN ANALYZE for both.")

Skipped on SQLite. On PostgreSQL this file times a monthly rollup before and after an index on (city, slot_at) and prints EXPLAIN ANALYZE for both.


> **Interview angle.** A query over 1.3 million booking rows that groups by city and month is slow. What do you look at first, and what would an index on (city, slot_at) change?
>
> *What a good answer covers:* EXPLAIN ANALYZE to see whether it is a sequential scan, a sort, or a hash aggregate that is slow, and whether the estimate is far from the actual rows. An index on (city, slot_at) lets the planner read rows in group order or skip to a date range, replacing a full scan plus sort with an index scan; it costs write time and space, so you add it for the queries finance will run weekly, not for a one-off.

## 4 · SUM

The same numbers you computed in pandas now come out of the database. Where they differ, one of the two is wrong; find out which before you go further. Write the SQL that finance will re-run into `sql/06_<your_brief>.sql` and reference it from the README.

## Before you move on

- Which of your pandas numbers did you reproduce in SQL, and did they agree to the rupee?
- Which query would you hand to finance, and could they read it without you in the room?
- What is the one index you would add if this ran weekly, and why not three?